In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [2]:
import os
import sys
import subprocess
import time
import json
import re
import shutil
import threading
import signal
import atexit
import queue
import importlib

# 🛠️ GLOBAL CONFIGURATION
KAGGLE_MODE = os.path.exists('/kaggle/working') or os.path.exists('/kaggle/input')
VERBOSE_MODE = True
START_TIME = time.time()
MAX_KAGGLE_SECONDS = 32400
EXPERIMENTAL_MODE = False

def vprint(*args, **kwargs):
    print(*args, **kwargs, flush=True)

# ==========================================
# 📂 VERIFIED KAGGLE DATASET PATHS
# ==========================================
# These are the EXACT paths from the Kaggle filesystem.
COMP_PATH          = "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3"
VLLM_WHEELS_PATH   = "/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/"
Z3_WHEELS_PATH     = "/kaggle/input/datasets/liquidvisualsinteractive/z3-solver-wheels"
MODEL_PATH_PRIMARY = "/kaggle/input/datasets/liquidvisualsinteractive/maa-deepseek-qwen-14b-ties-merged"
INTELLIB_PATH      = "/kaggle/input/datasets/liquidvisualsinteractive/maa-14b-core-intellib"
MODERNBERT_PATH    = "/kaggle/input/datasets/liquidvisualsinteractive/modernbert-base"

# ==========================================
# 🛡️ PRE-EMPTIVE SUBMISSION SKELETON
# ==========================================
def create_submission_skeleton():
    """Creates a valid submission.parquet immediately on boot as a safety net."""
    sample_path = os.path.join(COMP_PATH, "sample_submission.csv")
    try:
        if os.path.exists(sample_path):
            import pandas as pd
            sample = pd.read_csv(sample_path)
            sample.to_parquet('submission.parquet', index=False)
            vprint("📁 [PRE-EMPTIVE]: Submission skeleton created from sample.")
        else:
            # Fallback for local testing
            try:
                import polars as pl
                pl.DataFrame({'id': ['0'], 'answer': [0]}).write_parquet('submission.parquet')
                vprint("📁 [PRE-EMPTIVE]: Dummy submission.parquet created.")
            except: pass
    except Exception as e:
        vprint(f"⚠️ [PRE-EMPTIVE]: Failed to create skeleton: {e}")

if KAGGLE_MODE:
    create_submission_skeleton()

# ==========================================
# 🛰️ BOOTSTRAP UTILITIES: Shard & Tokenizer
# ==========================================

def patch_tokenizer(model_path):
    """Kaggle Fix: Patches tokenizer_config.json to handle broken backends and ChatML."""
    working_root = "/kaggle/working" if os.path.exists("/kaggle/working") else "./"
    tmp_tokenizer_path = os.path.join(working_root, f"fixed_tokenizer_{os.path.basename(model_path)}")

    if not os.path.exists(tmp_tokenizer_path):
        os.makedirs(tmp_tokenizer_path, exist_ok=True)
    
    try:
        found_files = os.listdir(model_path)
        for f in found_files:
            src_file = os.path.join(model_path, f)
            is_weight = f.endswith('.safetensors') or f.endswith('.bin') or f.endswith('.pt') or 'model-' in f
            if os.path.isfile(src_file) and not is_weight:
                try: shutil.copy2(src_file, tmp_tokenizer_path)
                except: pass
    except Exception as e:
        vprint(f"⚠️ Warning: Tokenizer patch preparation failed: {e}")

    config_path = os.path.join(tmp_tokenizer_path, "tokenizer_config.json")
    if os.path.exists(config_path):
        try:
            with open(config_path, 'r') as f:
                data = json.load(f)
            
            chat_template = (
                "{% for message in messages %}"
                "{% if message['role'] == 'user' %}"
                "{{ '<|im_start|>user\\n' + message['content'] + '<|im_end|>\\n' }}"
                "{% elif message['role'] == 'assistant' %}"
                "{{ '<|im_start|>assistant\\n' + message['content'] + '<|im_end|>\\n' }}"
                "{% endif %}"
                "{% endfor %}"
                "{% if add_generation_prompt %}"
                "{{ '<|im_start|>assistant\\n' }}"
                "{% endif %}"
            )
            data["chat_template"] = chat_template

            if data.get("tokenizer_class") == "TokenizersBackend":
                vprint(f"🔥 [CRITICAL FIX] Patching broken TokenizersBackend...")
                data.pop("tokenizer_class", None) 
            
            with open(config_path, 'w') as f:
                json.dump(data, f, indent=4)
        except Exception as e:
            vprint(f"⚠️ Warning: Failed to patch tokenizer config: {e}")

    return tmp_tokenizer_path

def _fix_kaggle_weight_filenames(model_path: str) -> str:
    """🔥 ADVANCED ARCHIVIST: Reconstructs Safetensors index to match Kaggle renames."""
    index_path = os.path.join(model_path, "model.safetensors.index.json")
    if not os.path.isfile(index_path): return model_path

    try:
        with open(index_path, 'r') as f:
            index_data = json.load(f)
    except: return model_path

    expected_files = sorted(set(index_data.get("weight_map", {}).values()))
    actual_safetensors = sorted([f for f in os.listdir(model_path) if f.endswith('.safetensors')])

    if set(expected_files) == set(actual_safetensors):
        vprint(f"✅ [ARCHIVIST]: Weight filenames match index. {len(actual_safetensors)} shards ready.")
        return model_path

    vprint("⚠️ [ARCHIVIST]: Kaggle filename mismatch detected. Repairing index...")
    
    actual_by_num = {}
    for f in actual_safetensors:
        digits = re.findall(r'\d+', f)
        if digits: actual_by_num[int(digits[0])] = f

    expected_by_num = {}
    for f in expected_files:
        digits = re.findall(r'\d+', f)
        if digits: expected_by_num[int(digits[0])] = f

    rename_map = {}
    for num in expected_by_num:
        if num in actual_by_num:
            rename_map[expected_by_num[num]] = actual_by_num[num]

    if not rename_map or len(rename_map) < len(expected_by_num):
        vprint("⚠️ [ARCHIVIST]: Fuzzy match incomplete. Using lexicographic fallback...")
        for exp, act in zip(sorted(expected_files), sorted(actual_safetensors)):
            rename_map[exp] = act

    fixed_path = os.path.join("/kaggle/working", "fixed_weights", os.path.basename(model_path.rstrip("/")))
    os.makedirs(fixed_path, exist_ok=True)

    for item in os.listdir(model_path):
        src, dst = os.path.join(model_path, item), os.path.join(fixed_path, item)
        if os.path.isfile(src) and not os.path.exists(dst):
            try: 
                os.symlink(src, dst)
            except: 
                if os.path.getsize(src) < 100*1024*1024: 
                    try: shutil.copy2(src, dst)
                    except: pass
                else:
                    vprint(f"⚠️ [ARCHIVIST]: Fallback copy blocked for large shard: {item}")

    new_weight_map = {k: rename_map.get(v, v) for k, v in index_data["weight_map"].items()}
    index_data["weight_map"] = new_weight_map

    repaired_index = os.path.join(fixed_path, "model.safetensors.index.json")
    if os.path.islink(repaired_index): os.unlink(repaired_index)
    with open(repaired_index, 'w') as f:
        json.dump(index_data, f)

    vprint(f"📦 [ARCHIVIST]: Shard mapping repaired. Weights at {fixed_path}")
    return fixed_path

# ==========================================
# 🚀 BOOTSTRAP PHASE 1: DEPENDENCY SYNC
# ==========================================
# This MUST run before any heavy imports (torch, vllm).
# Mirrors the proven lvties7ecopy.py boot sequence exactly.

def bootstrap_dependencies():
    """Phase 1: Install vLLM + Z3 wheels and register competition API path."""
    vprint("\n⚖️ [AUDIT]: AIMO-3 COMPLIANCE VERIFIED")
    vprint(f"   • Hardware Ceiling: 32GB RAM / 80GB VRAM (H100)")
    vprint(f"   • Time Limit: {MAX_KAGGLE_SECONDS}s")

    if not KAGGLE_MODE:
        vprint("[BOOTSTRAP]: Local mode — skipping dependency sync.")
        return

    vprint("[BOOTSTRAP]: Synchronizing Environment (Vanguard-Mode)...")

    # ===== STEP 0: Register competition API path =====
    # The kaggle_evaluation module lives inside the competition dataset.
    if os.path.exists(COMP_PATH) and COMP_PATH not in sys.path:
        sys.path.insert(0, COMP_PATH)
        vprint(f"✅ [PATH]: Competition API registered: {COMP_PATH}")

    # ===== STEP 1: Protobuf Sticky Patch =====
    try:
        import google.protobuf.message_factory as mf
        from google.protobuf import symbol_database
        def patch_factory(fi):
            if not hasattr(fi, 'GetPrototype'):
                fi.GetPrototype = lambda desc: symbol_database.Default().GetSymbol(desc.full_name)
        _orig = mf.MessageFactory.__init__
        def _patched(self, *a, **kw):
            _orig(self, *a, **kw)
            patch_factory(self)
        mf.MessageFactory.__init__ = _patched
        vprint("🧱 [PATCH]: Protobuf bridge established.")
    except: pass

    # ===== STEP 2: Install vLLM wheels =====
    if os.path.exists(VLLM_WHEELS_PATH):
        try:
            import vllm
            vprint(f"✅ [SENSE]: vLLM {vllm.__version__} already available.")
        except ImportError:
            vprint("🚀 [BOOTSTRAP]: Installing vLLM from local wheels...")
            import glob
            all_whls = sorted(glob.glob(os.path.join(VLLM_WHEELS_PATH, "**", "*.whl"), recursive=True))
            vprint(f"   Found {len(all_whls)} wheel files. Installing 1 by 1:")
            
            find_links = f"--find-links={VLLM_WHEELS_PATH}"
            installed = 0
            skipped = 0
            for i, whl_file in enumerate(all_whls, 1):
                basename = os.path.basename(whl_file)
                if any(x in basename.lower() for x in ["protobuf", "pillow", "numpy"]): 
                    vprint(f"   [{i}/{len(all_whls)}] ⏭️  SKIP: {basename}")
                    skipped += 1
                    continue
                vprint(f"   [{i}/{len(all_whls)}] 📦 Installing: {basename}")
                os.system(f"pip install --no-index --no-deps {find_links} --force-reinstall {whl_file} > /dev/null 2>&1")
                installed += 1
            vprint(f"   ✅ Installed: {installed} | Skipped: {skipped}")

            # Force environment refresh
            import site
            importlib.invalidate_caches()
            site.main()

            try:
                import vllm
                vprint(f"✅ [BOOTSTRAP]: vLLM {vllm.__version__} installed.")
            except ImportError as e:
                vprint(f"❌ [BOOTSTRAP ERROR]: vLLM not importable: {e}")
    else:
        vprint(f"⚠️ [BOOTSTRAP]: vLLM wheels not found at {VLLM_WHEELS_PATH}")

    # ===== STEP 3: Install Z3 solver =====
    if os.path.exists(Z3_WHEELS_PATH):
        try:
            import z3
            vprint(f"✅ [SENSE]: Z3 solver already available.")
        except ImportError:
            vprint("📦 [BOOTSTRAP]: Installing Z3 solver...")
            import glob
            z3_whls = glob.glob(os.path.join(Z3_WHEELS_PATH, "*.whl"))
            for whl in z3_whls:
                os.system(f"pip install --no-index --no-deps {whl} > /dev/null 2>&1")
            importlib.invalidate_caches()
            try:
                import z3
                vprint(f"✅ [BOOTSTRAP]: Z3 solver installed.")
            except ImportError:
                vprint(f"⚠️ [BOOTSTRAP]: Z3 install failed (non-critical).")

    # ===== STEP 4: Verify all dataset paths =====
    vprint("\n📂 [DATASET AUDIT]:")
    datasets = {
        "Competition API": COMP_PATH,
        "vLLM Wheels":     VLLM_WHEELS_PATH,
        "Z3 Wheels":       Z3_WHEELS_PATH,
        "14B Model":       MODEL_PATH_PRIMARY,
        "IntelLib":        INTELLIB_PATH,
        "ModernBERT":      MODERNBERT_PATH,
    }
    for name, path in datasets.items():
        exists = os.path.exists(path)
        icon = "✅" if exists else "❌"
        vprint(f"   {icon} {name}: {path}")

    vprint("\n✅ [BOOTSTRAP COMPLETE]: FOUNDATION READY.\n")

# 🔥 TRIGGER DEPENDENCY SYNC IMMEDIATELY (before any heavy imports)
bootstrap_dependencies()

# ==========================================
# 🧱 POST-BOOTSTRAP IMPORTS (Safe to load now)
# ==========================================
import polars as pl
import numpy as np
import torch
torch.set_default_dtype(torch.bfloat16)

try:
    from vllm import LLM, SamplingParams
    vprint(f"✅ [IMPORT]: vLLM LLM & SamplingParams loaded.")
except ImportError:
    LLM = None
    SamplingParams = None
    vprint("⚠️ [IMPORT]: vLLM not available. Running in mock mode.")

# 🚨 SYSTEM STABILITY FLAGS
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
os.environ["RAY_LOG_TO_STDERR"] = "0"
os.environ["RAY_IGNORE_UNHANDLED_ERRORS"] = "1"
os.environ["NCCL_LOG_LEVEL"] = "ERROR"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# ==========================================
# 📚 PRE-LOAD: IntelLib Knowledge Base
# ==========================================
INTELLIB_DATA = {}
if os.path.exists(INTELLIB_PATH):
    import glob as _glob
    for pq_file in _glob.glob(os.path.join(INTELLIB_PATH, "*.parquet")):
        key = os.path.splitext(os.path.basename(pq_file))[0]  # e.g. "axioms", "theorems"
        try:
            INTELLIB_DATA[key] = pl.read_parquet(pq_file)
            vprint(f"✅ [INTELLIB]: Loaded {key} ({len(INTELLIB_DATA[key])} rows)")
        except Exception as e:
            vprint(f"⚠️ [INTELLIB]: Failed to load {key}: {e}")
    vprint(f"📚 [INTELLIB]: {len(INTELLIB_DATA)} knowledge tables ready: {list(INTELLIB_DATA.keys())}")
else:
    vprint("⚠️ [INTELLIB]: Path not found, skipping.")

# ==========================================
# 🧠 PRE-LOAD: ModernBERT Path Verification
# ==========================================
MODERNBERT_READY = False
if os.path.exists(MODERNBERT_PATH):
    _expected = ["config.json", "tokenizer.json", "model.safetensors"]
    _found = [f for f in _expected if os.path.exists(os.path.join(MODERNBERT_PATH, f))]
    MODERNBERT_READY = len(_found) == len(_expected)
    vprint(f"✅ [MODERNBERT]: Model verified ({len(_found)}/{len(_expected)} files). Path: {MODERNBERT_PATH}")
else:
    vprint("⚠️ [MODERNBERT]: Path not found, skipping.")

# ==========================================
# 📋 RESOURCE MANIFEST (Bootstrap Summary)
# ==========================================
vprint("\n" + "="*70)
vprint("📋 RESOURCE MANIFEST — Everything loaded at bootstrap:")
vprint("="*70)
vprint(f"  1. vLLM Engine     : {'✅ Installed' if LLM is not None else '❌ Not available'}")
try:
    import vllm as _vllm_check
    vprint(f"     Version         : {_vllm_check.__version__}")
except: pass
# vLLM wheel count (already listed during install)
if os.path.exists(VLLM_WHEELS_PATH):
    import glob as _g
    _whls = sorted(_g.glob(os.path.join(VLLM_WHEELS_PATH, "*.whl")))
    vprint(f"     Wheels          : {len(_whls)} installed")
try:
    import z3 as _z3_check
    vprint(f"  2. Z3 Solver       : ✅ Installed")
except:
    vprint(f"  2. Z3 Solver       : ❌ Not available")
vprint(f"  3. Competition API : {'✅ ' + COMP_PATH if COMP_PATH in sys.path else '❌ Not in sys.path'}")
# List 14B model shards
vprint(f"  4. 14B Model Shards: {'✅ ' + MODEL_PATH_PRIMARY if os.path.exists(MODEL_PATH_PRIMARY) else '❌ Not found'}")
if os.path.exists(MODEL_PATH_PRIMARY):
    _model_files = sorted(os.listdir(MODEL_PATH_PRIMARY))
    _shards = [f for f in _model_files if f.endswith('.safetensors')]
    _configs = [f for f in _model_files if not f.endswith('.safetensors')]
    _total_gb = sum(os.path.getsize(os.path.join(MODEL_PATH_PRIMARY, f)) for f in _shards) / (1024**3)
    vprint(f"     Configs         : {_configs}")
    vprint(f"     Shards ({len(_shards)}, {_total_gb:.2f} GB):")
    for _s in _shards:
        _sz = os.path.getsize(os.path.join(MODEL_PATH_PRIMARY, _s)) / (1024**2)
        vprint(f"       • {_s} ({_sz:.1f} MB)")
vprint(f"  5. IntelLib Tables : {'✅ ' + str(list(INTELLIB_DATA.keys())) if INTELLIB_DATA else '❌ Empty'}")
vprint(f"  6. ModernBERT      : {'✅ Ready' if MODERNBERT_READY else '❌ Not verified'}")
vprint(f"  7. PyTorch         : ✅ {torch.__version__} (dtype: {torch.get_default_dtype()})")
if torch.cuda.is_available():
    vprint(f"  8. GPU             : ✅ {torch.cuda.get_device_name(0)}")
    vprint(f"     VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    vprint(f"  8. GPU             : ❌ No CUDA device")
vprint("="*70 + "\n")

# ==========================================
# 🏁 FOUNDATION: 14B vLLM Resource Wrapper
# ==========================================

class ModelOrchestrator:
    def __init__(self, model_path):
        self.model_path = model_path
        self.llm = None
        self._load_model(model_path)

    def _load_model(self, model_path):
        """Loads vLLM + 14B shards into VRAM. Single point of loading."""
        if model_path == "mock" or LLM is None:
            vprint("🧪 [LLM]: Mock Engine Active (no GPU or vLLM unavailable).")
            return

        # ===== STEP 1: Fix shard filenames =====
        fixed_path = _fix_kaggle_weight_filenames(model_path)

        # ===== STEP 2: Patch tokenizer =====
        patched_tokenizer = patch_tokenizer(fixed_path)

        # ===== STEP 3: Shard Scout =====
        import glob
        shards = sorted(glob.glob(os.path.join(fixed_path, "*.safetensors")))
        if not shards:
            shards = sorted(glob.glob(os.path.join(fixed_path, "*.bin")))
        
        total_gb = sum(os.path.getsize(f) for f in shards) / (1024**3)
        vprint(f"\n🛰️ [SHARD SCOUT]: Detected {len(shards)} weight shards ({total_gb:.2f} GB)")
        vprint(f"   Model path: {fixed_path}")
        for i, f in enumerate(shards):
            vprint(f"   |-- [SHARD {i+1}/{len(shards)}]: {os.path.basename(f)} ({os.path.getsize(f)/(1024**2):.1f} MB)")
        
        if total_gb < 5.0:
            vprint("⚠️ [CRITICAL WARNING]: Weight volume < 5GB! Model may be incomplete.")

        # ===== STEP 4: VRAM Telemetry =====
        stop_monitor = threading.Event()
        def monitor_vram():
            t0 = time.time()
            vprint("📟 [VRAM]: Materialization monitoring active...")
            while not stop_monitor.is_set():
                if torch.cuda.is_available():
                    free, total = torch.cuda.mem_get_info(0)
                    used = (total - free) / (1024**3)
                    total_gb = total / (1024**3)
                    sys.stdout.write(f"\r📟 [VRAM]: Occupancy: {used:.2f} GB / {total_gb:.0f} GB [ {int(time.time()-t0)}s ]")
                    sys.stdout.flush()
                time.sleep(2)
            print()

        monitor_thread = threading.Thread(target=monitor_vram, daemon=True)
        monitor_thread.start()

        # ===== STEP 5: Create vLLM Engine =====
        try:
            old_log = os.environ.get("VLLM_LOGGING_LEVEL", "WARNING")
            os.environ["VLLM_LOGGING_LEVEL"] = "INFO"

            vprint(f"🚀 [LOADER]: Initializing 14B Engine (0.91 VRAM, 16K ctx)...")
            self.llm = LLM(
                model=fixed_path,
                tokenizer=patched_tokenizer,
                tensor_parallel_size=1,
                dtype="bfloat16",
                enable_prefix_caching=True,
                disable_log_stats=True,
                enforce_eager=False,
                gpu_memory_utilization=0.91,
                max_model_len=16384,
                trust_remote_code=True,
                disable_custom_all_reduce=True,
            )
            vprint(f"✅ [LLM]: 14B Model fully materialized in VRAM.")
            os.environ["VLLM_LOGGING_LEVEL"] = old_log

        except Exception as e:
            vprint(f"🧨 [LLM ERROR]: Materialization failed: {e}")
            import traceback
            traceback.print_exc()
        finally:
            stop_monitor.set()
            monitor_thread.join()

        # ===== STEP 6: Post-load VRAM verification =====
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info(0)
            vram_used = (total - free) / (1024**3)
            vprint(f"📟 [VRAM FINAL]: {vram_used:.2f} GB occupied.")
            if vram_used > 20:
                vprint("✅ [VRAM]: Shards confirmed in GPU memory.")
            else:
                vprint("⚠️ [VRAM]: Occupancy suspiciously low — shards may not have loaded.")

    def predict(self, prompt: str) -> str:
        if not self.llm: return "0"
        try:
            params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=2048, stop=["<|im_end|>"])
            outputs = self.llm.generate([prompt], params, use_tqdm=False)
            return outputs[0].outputs[0].text
        except Exception as e:
            vprint(f"⚠️ [INFERENCE ERROR]: {e}")
            return "0"


class Model:
    """Submission wrapper. Loads resources eagerly, solves with zero-baseline."""
    def __init__(self):
        self.orchestrator = None

    def load(self):
        if self.orchestrator is not None:
            return

        model_path = "mock"
        if os.path.exists(MODEL_PATH_PRIMARY):
            model_path = MODEL_PATH_PRIMARY
            vprint(f"✅ [MODEL FOUND]: {model_path}")
        else:
            vprint("⚠️ [MODEL]: No model dataset found. Running mock mode.")
        
        self.orchestrator = ModelOrchestrator(model_path)

    def solve(self, problem_text: str) -> int:
        if self.orchestrator is None: self.load()
        vprint(f"📝 [SOLVE]: Problem ({len(problem_text)} chars). Returning baseline 0.")
        return 0


model = Model()


def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
    """Kaggle API Gateway."""
    id_val = id_.item(0) if hasattr(id_, "item") else id_[0]
    problem_text = problem.item(0) if hasattr(problem, "item") else problem[0]
    vprint(f"\n📥 [REQUEST]: ID {id_val}")
    
    final_answer = 0
    try:
        final_answer = model.solve(problem_text)
        final_answer = int(float(str(final_answer))) % 1000
    except:
        final_answer = 0
    
    vprint(f"✅ [RESULT]: {final_answer} | ID: {id_val}")
    return pl.DataFrame({'id': id_, 'answer': pl.Series([final_answer], dtype=pl.Int64)})


def run_diagnostic_suite():
    vprint("\n☢️ [DIAGNOSTIC]: VERIFYING RESOURCE FOUNDATION")
    model.load()
    
    if model.orchestrator and model.orchestrator.llm:
        vprint("🚀 [DIAGNOSTIC]: Testing inference...")
        resp = model.orchestrator.predict("Say 'Ready'.")
        vprint(f"✅ [LLM]: Response: {resp.strip()[:100]}")
        
        vram_used = torch.cuda.memory_allocated() / (1024**3)
        vprint(f"📟 [VRAM]: {vram_used:.2f} GB occupied.")
    else:
        vprint("⚠️ [DIAGNOSTIC]: LLM not loaded (mock mode or failure).")
    
    vprint("✅ [DIAGNOSTIC COMPLETE]\n")


# ==========================================
# 🏁 MAIN ENTRY POINT
# ==========================================
if __name__ == "__main__":
    if EXPERIMENTAL_MODE:
        # Experiment mode: load model, run diagnostics, then stop.
        vprint("\n🧪 [EXPERIMENT]: Running diagnostic suite only...")
        model.load()
        run_diagnostic_suite()
        vprint("🧪 [EXPERIMENT]: Done. Exiting.")
    elif KAGGLE_MODE:
        # Production mode: load model, start inference server.
        vprint("\n🚀 [KAGGLER]: Eager Model Materialization starting...")
        model.load()
        
        import kaggle_evaluation.aimo_3_inference_server
        vprint("📡 [SERVER]: Initializing AIMO 3 Inference Server...")
        server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
        
        vprint("\n" + "="*70)
        vprint("[READY]: Inference Server listening. Standing by for problems...")
        vprint("=" * 70 + "\n")
        
        server.serve()
    else:
        vprint("[DEV]: Local mock test.")
        model.load()
        run_diagnostic_suite()


📁 [PRE-EMPTIVE]: Submission skeleton created from sample.

⚖️ [AUDIT]: AIMO-3 COMPLIANCE VERIFIED
   • Hardware Ceiling: 32GB RAM / 80GB VRAM (H100)
   • Time Limit: 32400s
[BOOTSTRAP]: Synchronizing Environment (Vanguard-Mode)...
🧱 [PATCH]: Protobuf bridge established.
🚀 [BOOTSTRAP]: Installing vLLM from local wheels...
   Found 163 wheel files. Installing 1 by 1:
   [1/163] 📦 Installing: aiohappyeyeballs-2.6.1-py3-none-any.whl
   [2/163] 📦 Installing: aiohttp-3.13.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
   [3/163] 📦 Installing: aiosignal-1.4.0-py3-none-any.whl
   [4/163] 📦 Installing: annotated_doc-0.0.4-py3-none-any.whl
   [5/163] 📦 Installing: annotated_types-0.7.0-py3-none-any.whl
   [6/163] 📦 Installing: anthropic-0.84.0-py3-none-any.whl
   [7/163] 📦 Installing: anyio-4.12.1-py3-none-any.whl
   [8/163] 📦 Installing: apache_tvm_ffi-0.1.8.post2-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
   [9/163] 📦 Installing: astor-0.8.1-

2026-04-11 15:20:20.499600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775920820.803042      43 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775920820.892733      43 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775920821.673222      43 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775920821.673260      43 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775920821.673263      43 computation_placer.cc:177] computation placer alr

✅ [IMPORT]: vLLM LLM & SamplingParams loaded.
✅ [INTELLIB]: Loaded axioms (1005 rows)
✅ [INTELLIB]: Loaded theorems (500000 rows)
✅ [INTELLIB]: Loaded definitions (150000 rows)
✅ [INTELLIB]: Loaded strategies (1 rows)
📚 [INTELLIB]: 4 knowledge tables ready: ['axioms', 'theorems', 'definitions', 'strategies']
✅ [MODERNBERT]: Model verified (3/3 files). Path: /kaggle/input/datasets/liquidvisualsinteractive/modernbert-base

📋 RESOURCE MANIFEST — Everything loaded at bootstrap:
  1. vLLM Engine     : ✅ Installed
     Version         : 0.16.0
     Wheels          : 163 installed
  2. Z3 Solver       : ✅ Installed
  3. Competition API : ✅ /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3
  4. 14B Model Shards: ✅ /kaggle/input/datasets/liquidvisualsinteractive/maa-deepseek-qwen-14b-ties-merged
     Configs         : ['config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json']
     Shards (6, 27.51 GB):
       • model-00001-of-00006.safetensors 

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


📟 [VRAM]: Occupancy: 0.51 GB / 79 GB [ 0s ]

...

📟 [VRAM]: Occupancy: 68.14 GB / 79 GB [ 384s ]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 16.68it/s]
Capturing CUDA graphs (decode, FULL):  94%|█████████▍| 48/51 [00:01<00:00, 28.85it/s]

📟 [VRAM]: Occupancy: 68.54 GB / 79 GB [ 386s ]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 26.95it/s]


INFO 04-11 15:27:21 [llm.py:355] Supported tasks: ['generate']
✅ [LLM]: 14B Model fully materialized in VRAM.

📟 [VRAM FINAL]: 74.18 GB occupied.
✅ [VRAM]: Shards confirmed in GPU memory.
📡 [SERVER]: Initializing AIMO 3 Inference Server...

[READY]: Inference Server listening. Standing by for problems...

